In [14]:
import os
print(os.getcwd())                   # 현재 작업 폴더
print(os.path.exists("1718848317_Sample_1.mp3"))

c:\workspace\archive\chap05
True


In [24]:
%pip install --upgrade pip
%pip install --upgrade transformers datasets[audio] accelerate

Note: you may need to restart the kernel to use updated packages.
  Using cached datasets-4.0.0-py3-none-any.whl.metadata (19 kB)
INFO: pip is looking at multiple versions of datasets[audio] to determine which version is compatible with other requirements. This could take a while.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
os.environ["PATH"] += os.pathsep + r"C:\workspace\archive\ffmpeg-2025-08-18-git-0226b6fb2c-essentials_build\bin"

In [4]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3-turbo"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
  model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True,
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
pipe = pipeline(
  "automatic-speech-recognition",
  model=model,
  tokenizer=processor.tokenizer,
  feature_extractor=processor.feature_extractor,
  torch_dtype=torch_dtype,
  device=device,
  return_timestamps=True,
  chunk_length_s=10,    
  stride_length_s=2,
)

sample = "광화문자생한방병원_1.mp3"

result = pipe(sample, generate_kwargs={"language": "ko"})
print(result["text"])

Device set to use cpu
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


ValueError: ffmpeg was not found but is required to load audio files from filename

In [7]:
start_end_text = []

for chunk in result["chunks"]:
  start = chunk["timestamp"][0]
  end = chunk["timestamp"][1]
  text = chunk["text"]
  start_end_text.append([start, end, text])
  
import pandas as pd
df = pd.DataFrame(start_end_text, columns=["start", "end", "text"])
df.to_csv("lsy_audio_2023_58.csv", index=False, sep="|", encoding="utf-8-sig")
display(df)

,start,end,text
0,0.0,6.72,"척추 디스크 비수술 치료, 광화문 자생한방병원으로 가실 분은 서대문역 6번 출구로..."
